### **Package**

In [28]:
import subprocess
import sys
import os

print(sys.version)

try:
    import pymoo
    print("pymoo is already installed.")
except ImportError:
    print("pymoo is not installed. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pymoo"])
print("pymoo installed.")

try:
    import pyro
    print("Pyro is already installed.")
except ImportError:
    print("Pyro is not installed. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyro-ppl"])
    print("Pyro installed.")


import warnings
warnings.filterwarnings("ignore", message=".*load_learner.*pickle.*")

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
pymoo is already installed.
pymoo installed.
Pyro is already installed.


In [29]:
import numpy as np
import pandas as pd
import time
import random
import matplotlib.pyplot as plt
from pymoo.operators.sampling.lhs import LHS
from pymoo.problems import get_problem
from pymoo.core.problem import Problem
from pymoo.util.ref_dirs import get_reference_directions
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler


# Optimization algorithm
from pymoo.optimize import minimize
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.termination import get_termination
from pymoo.core.survival import Survival
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from pymoo.operators.survival.rank_and_crowding.metrics import get_crowding_function
from pymoo.util.randomized_argsort import randomized_argsort
from pymoo.problems.multi.omnitest import OmniTest


# Surrogate model
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.optim import Adam


# Metrics
from pymoo.indicators.hv import HV
from pymoo.indicators.igd_plus import IGDPlus
from sklearn.metrics import mean_squared_error, mean_absolute_error

### **Class and Function**

###### Plot and Metrics

In [30]:
# Plot: 2 Objs and pareto front
def plot_obj_2d(F, xlim=(0, 1), ylim=(0, 1),):
    n_obj = F.shape[1]
    if n_obj == 2:
        nds = NonDominatedSorting()
        front_idx = nds.do(F, only_non_dominated_front=True)

        pareto_F = F[front_idx]
        non_pareto_F = np.delete(F, front_idx, axis=0)

        fig = go.Figure(
            data=go.Scatter(
                x=F[:, 0],
                y=F[:, 1],
                mode='markers',
                name='Objective Values',
                marker=dict(size=6, color='#87CEEB', opacity=0.7)
            )
        )
        fig.add_trace(go.Scatter(
            x=pareto_F[:, 0],
            y=pareto_F[:, 1],
            mode='markers',
            name='Pareto Front',
            marker=dict(size=7, color='#FF7F0E', opacity=0.9, symbol='diamond')
    ))
        fig.update_layout(
            xaxis_title='f1',
            yaxis_title='f2',
            width=600,
            height=600,
            xaxis=dict(range=list(xlim)),
            yaxis=dict(range=list(ylim))
        )
    fig.show()


def mean_std(arr):
    return np.mean(arr), np.std(arr)

In [31]:
def evaluate_pre_real(pre, real, title=None, figsize=(7, 6), point_size=20, show_plot=True):
    pre = np.asarray(pre, dtype=float)
    real = np.asarray(real, dtype=float)

    if pre.ndim != 2 or real.ndim != 2:
        raise ValueError("pre and real must be 2D arrays.")
    if pre.shape[1] != 2 or real.shape[1] != 2:
        raise ValueError("pre and real must have shape (n, 2).")
    if pre.shape[0] != real.shape[0]:
        raise ValueError("pre and real must have the same number of rows.")

    # row-wise Euclidean distance
    distances = np.sqrt(np.sum((pre - real) ** 2, axis=1))

    max_idx = np.argmax(distances)
    min_idx = np.argmin(distances)

    result = {
        "distances": distances,
        "max_distance": distances[max_idx],
        "max_obj_point": pre[max_idx],
        "max_f_real_point": real[max_idx],
        "min_distance": distances[min_idx],
        "min_obj_point": pre[min_idx],
        "min_f_real_point": real[min_idx],
        "mean_distance": np.mean(distances)
    }

    if show_plot:
        fig, ax = plt.subplots(figsize=figsize)

        for i in range(pre.shape[0]):
            ax.annotate(
                '',
                xy=(real[i, 0], real[i, 1]),
                xytext=(pre[i, 0], pre[i, 1]),
                arrowprops=dict(
                    arrowstyle='->',
                    color='green',
                    lw=1.0,
                    alpha=0.8,
                    shrinkA=0,
                    shrinkB=0
                )
            )

        ax.scatter(
            pre[:, 0], pre[:, 1],
            color='#87CEEB',
            s=point_size,
            alpha=0.8,
            label='pre'
        )

        ax.scatter(
            real[:, 0], real[:, 1],
            color='#FF7F0E',
            s=point_size,
            alpha=0.8,
            label='real'
        )

        ax.set_xlabel('F1')
        ax.set_ylabel('F2')
        if title is not None:
            ax.set_title(title)
        ax.legend()
        plt.tight_layout()
        plt.show()

    print(f"Max:  {result['max_distance']:.2f}, sur={result['max_obj_point']}, real={result['max_f_real_point']}")
    print(f"Min:  {result['min_distance']:.2f}, sur={result['min_obj_point']}, real={result['min_f_real_point']}")
    print(f"Mean: {result['mean_distance']:.2f}")
    print("-" * 50)

    return result

###### Class: Surrogate model

In [32]:
class BNN(PyroModule):
    def __init__(self, in_dim, hidden=32, out_dim=1, fixed_sigma=1e-3):
        super().__init__()

        # Bayesian Linear Layer 1
        self.fc1 = PyroModule[nn.Linear](in_dim, hidden)
        self.fc1.weight = PyroSample(
            dist.Normal(0., 0.5).expand([hidden, in_dim]).to_event(2))
        self.fc1.bias = PyroSample(
            dist.Normal(0., 0.5).expand([hidden]).to_event(1))

        # Bayesian Linear Layer 2
        self.fc2 = PyroModule[nn.Linear](hidden, hidden)
        self.fc2.weight = PyroSample(
            dist.Normal(0., 0.5).expand([hidden, hidden]).to_event(2))
        self.fc2.bias = PyroSample(
            dist.Normal(0., 0.5).expand([hidden]).to_event(1))

        # Bayesian output layer
        self.out = PyroModule[nn.Linear](hidden, out_dim)
        self.out.weight = PyroSample(
            dist.Normal(0., 0.5).expand([out_dim, hidden]).to_event(2))
        self.out.bias = PyroSample(
            dist.Normal(0., 0.5).expand([out_dim]).to_event(1))

        self.sigma = fixed_sigma

    def forward(self, x, y=None):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))

        mu = self.out(x).squeeze(-1)
        pyro.deterministic("mean", mu)

        with pyro.plate("data", x.shape[0]):
            pyro.sample("obs", dist.Normal(mu, self.sigma), obs=y)

        return mu



@torch.no_grad()
def predict_bnn(model, guide, X_test, num_samples=100):
    X_test = torch.tensor(X_test, dtype=torch.float32)
    predictive = Predictive(model, guide=guide, num_samples=num_samples)
    samples = predictive(X_test)

    mu_samples = samples["mean"] # epistemic uncertainty
    mu_mean = mu_samples.mean(dim=0).cpu().numpy().reshape(-1, 1)

    obs_samples = samples["obs"].cpu().numpy() # total uncertainty
    y_q80 = np.percentile(obs_samples, 80, axis=0).reshape(-1, 1)
    y_q90 = np.percentile(obs_samples, 90, axis=0).reshape(-1, 1)
    y_q95 = np.percentile(obs_samples, 95, axis=0).reshape(-1, 1)

    return mu_mean, y_q80, y_q90, y_q95

In [33]:
def train_bnn(
    X_train, y_train,
    X_val, y_val,
    hidden=16,
    lr=1e-3,
    max_steps=50000,
    patience=10,
    device=None
):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    pyro.clear_param_store()

    model = BNN(in_dim=X_train.shape[1], hidden=hidden).to(device)
    guide = AutoDiagonalNormal(model)
    optimizer = Adam({"lr": lr})
    svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

    best_val = float("inf")
    no_improve = 0

    # -------- train --------
    for step in range(1, max_steps + 1):
        loss = svi.step(X_train, y_train)

        if step % 200 == 0:
            # posterior predictive validation
            predictive = Predictive(model, guide=guide, num_samples=50)
            samples = predictive(X_val)
            mu = samples["mean"].mean(0)
            val_mse = ((mu - y_val) ** 2).mean().item()

            if val_mse < best_val:
                best_val = val_mse
                best_state = pyro.get_param_store().get_state()
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f"Early stopping at step {step}, best val={best_val:.2e}")
                break

            print(
                  f"[{step}] "
                  f"train ELBO={loss:.2e}, "
                  f"val MSE={val_mse:.2e}, "
                  f"no_improve={no_improve}/{patience}")

    return model, guide


###### Class: Problem

In [34]:
class Benchmark_Problem(Problem):
    def __init__(
        self,
        model_f1, model_f2,
        guide_f1, guide_f2,
        scaler_y_f1, scaler_y_f2,
        n_var, n_obj, xl, xu,
        problem_name,
        use_surrogate):

        if 'dtlz' in problem_name:
            self.problem = get_problem(problem_name, n_var=n_var, n_obj=n_obj)
        elif 'omnitest' in problem_name:
            self.problem = OmniTest(n_var=n_var)
        else:
            self.problem = get_problem(problem_name)

        n_constr = self.problem.n_constr if self.problem.has_constraints() else 0

        super().__init__(
            n_var=n_var,
            n_obj=n_obj,
            xl=xl,
            xu=xu,
            n_constr=n_constr,
            model=None,
            evaluation_model=None
        )

        # ---- store surrogate models ----
        self.model_f1 = model_f1
        self.model_f2 = model_f2
        self.guide_f1 = guide_f1
        self.guide_f2 = guide_f2
        self.scaler_y_f1 = scaler_y_f1
        self.scaler_y_f2 = scaler_y_f2
        self.use_surrogate = use_surrogate

    def _evaluate(self, X, out, *args, **kwargs):

        if self.use_surrogate == 'BNN_uncertainty':

            # ---- BNN predictive ----
            y1_q50, y1_q80, y1_q90, y1_q95 = predict_bnn(
                self.model_f1, self.guide_f1, X, num_samples=100
            )
            y2_q50, y2_q80, y2_q90, y2_q95 = predict_bnn(
                self.model_f2, self.guide_f2, X, num_samples=100
            )

            # ---- inverse scaling ----
            y1_q50 = self.scaler_y_f1.inverse_transform(y1_q50)
            y1_q80 = self.scaler_y_f1.inverse_transform(y1_q80)
            y1_q90 = self.scaler_y_f1.inverse_transform(y1_q90)
            y1_q95 = self.scaler_y_f1.inverse_transform(y1_q95)

            y2_q50 = self.scaler_y_f2.inverse_transform(y2_q50)
            y2_q80 = self.scaler_y_f2.inverse_transform(y2_q80)
            y2_q90 = self.scaler_y_f2.inverse_transform(y2_q90)
            y2_q95 = self.scaler_y_f2.inverse_transform(y2_q95)

            # ---- stack objectives ----
            out["F"] = np.hstack([y1_q50, y2_q50])
            out["F_q80"] = np.hstack([y1_q80, y2_q80])
            out["F_q90"] = np.hstack([y1_q90, y2_q90])
            out["F_q95"] = np.hstack([y1_q95, y2_q95])


###### Class: Survival_standard

In [35]:
class Survival_standard(Survival):

    def __init__(self, nds=None, crowding_func="cd"):
        crowding_func_ = get_crowding_function(crowding_func)
        super().__init__(filter_infeasible=True)
        self.nds = nds if nds is not None else NonDominatedSorting()
        self.crowding_func = crowding_func_


    def _do(self,
            problem,
            pop,
            *args,
            random_state=None,
            n_survive=None,
            **kwargs):

        F = pop.get("F").astype(float, copy=False)

        survivors = []

        fronts = self.nds.do(F, n_stop_if_ranked=n_survive)

        for k, front in enumerate(fronts):

            I = np.arange(len(front))

            if len(survivors) + len(I) > n_survive:
                n_remove = len(survivors) + len(front) - n_survive
                crowding_of_front = \
                    self.crowding_func.do(
                        F[front, :],
                        n_remove=n_remove
                    )

                I = randomized_argsort(crowding_of_front, order='descending', method='numpy', random_state=random_state)
                I = I[:-n_remove]

            else:
                crowding_of_front = \
                    self.crowding_func.do(
                        F[front, :],
                        n_remove=0
                    )

            for j, i in enumerate(front):
                pop[i].set("rank", k)
                pop[i].set("crowding", crowding_of_front[j])
            survivors.extend(front[I])

        return pop[survivors]


###### Class: Survival_fix_dual_ranking

In [36]:
class Survival_fix_dual_ranking(Survival):

    def __init__(self, nds=None, crowding_func="cd", alpha=1):
        crowding_func_ = get_crowding_function(crowding_func)
        super().__init__(filter_infeasible=True)
        self.nds = nds if nds is not None else NonDominatedSorting()
        self.crowding_func = crowding_func_
        self.alpha = alpha

    def _do(self,problem,pop,*args,random_state=None,n_survive=None,**kwargs):
        F = pop.get("F").astype(float, copy=False)
        F_q80 = pop.get("F_q80").astype(float, copy=False)
        F_q90 = pop.get("F_q90").astype(float, copy=False)
        F_q95 = pop.get("F_q95").astype(float, copy=False)

        #============= F_fix =====================
        if self.alpha == 0.8:
          F_hybrid = np.concatenate([F, F_q80], axis=1)
        elif self.alpha == 0.9:
          F_hybrid = np.concatenate([F, F_q90], axis=1)
        elif self.alpha == 0.95:
          F_hybrid = np.concatenate([F, F_q95], axis=1)
        #===========================================

        # ====== NonDominatedSorting ============
        fronts_hybrid = NonDominatedSorting().do(F_hybrid)
        #==========================================

        survivors = []
        for k, front in enumerate(fronts_hybrid):

            I = np.arange(len(front))

            if len(survivors) + len(I) > n_survive:
                n_remove = len(survivors) + len(front) - n_survive
                crowding_of_front = \
                    self.crowding_func.do(
                        F[front, :],
                        n_remove=n_remove
                    )

                I = randomized_argsort(crowding_of_front, order='descending', method='numpy', random_state=random_state)
                I = I[:-n_remove]

            else:
                crowding_of_front = \
                    self.crowding_func.do(
                        F[front, :],
                        n_remove=0
                    )

            for j, i in enumerate(front):
                pop[i].set("rank", k)
                pop[i].set("crowding", crowding_of_front[j])
            survivors.extend(front[I])

        return pop[survivors]


### **Main**

###### 1. Initial settings



In [37]:
# Initial settings
# Problem: dtlz1-7, omnitest, bnh, truss2d, welded_beam
problem_name = 'dtlz5'

if 'dtlz' in problem_name:
  n_var = 10
  n_obj = 2
  problem = get_problem(problem_name, n_var=n_var, n_obj=n_obj)

elif 'omnitest' in problem_name:
  n_var = 2
  n_obj = 2
  problem = OmniTest(n_var=n_var)

else:
  problem = get_problem(problem_name)
  n_var = problem.n_var
  n_obj = problem.n_obj

print(f"Problem name: {problem_name}")
print(f"Cons: {problem.n_constr}")
print(f"Var: {n_var}")
print(f"Obj: {n_obj}")

# n_gen, pop_size
n_gen = 100
pop_size = 100
np.set_printoptions(precision=4, suppress=True)

# Data
# LHS sampling with seed 42
np.random.seed(42)
sample_size = 11*n_var-1
sampling = LHS()

X_train = sampling(problem, sample_size, seed=42).get("X")
y_train = problem.evaluate(X_train, return_values_of=["F"])
y_train_f1 = y_train[:, 0]
y_train_f2 = y_train[:, 1]
print('\nSampling X_train shape: ', X_train.shape)

X_val = sampling(problem, 100, seed=42).get("X")
y_val = problem.evaluate(X_val, return_values_of=["F"])
print('y_val shape: ', y_val.shape)

X_test = sampling(problem, 100, seed=1).get("X")
y_test = problem.evaluate(X_test, return_values_of=["F"])
print('y_test shape: ', y_test.shape)


# Metrics: HV
if problem_name == 'dtlz1':
  obj_min = np.array([0,0])
  obj_max = np.array([552.30,568.36])

if problem_name == 'dtlz2':
  obj_min = np.array([0,0])
  obj_max = np.array([2.78,2.93])

if problem_name == 'dtlz3':
  obj_min = np.array([0,0])
  obj_max = np.array([1605.54,1670.48])

if problem_name == 'dtlz4':
  obj_min = np.array([0,0])
  obj_max = np.array([2.83,2.78])

if problem_name == 'dtlz5':
  obj_min = np.array([0,0])
  obj_max = np.array([2.61,2.70])

if problem_name == 'dtlz6':
  obj_min = np.array([0,0])
  obj_max = np.array([9.78,9.78])

if problem_name == 'dtlz7':
  obj_min = np.array([0,0])
  obj_max = np.array([1.10,33.43])

if problem_name == 'omnitest':
  obj_min = np.array([-2,-2])
  obj_max = np.array([2.40,2.40])

if problem_name == 'bnh':
  obj_min = np.array([0,5])
  obj_max = np.array([140,50])

if problem_name == 'truss2d':
  obj_min = np.array([0,0])
  obj_max = np.array([0.1,1e5])

if problem_name == 'welded_beam':
  obj_min = np.array([0,0])
  obj_max = np.array([160,0.20])


ref_point = np.array([1.1,1.1])
hv = HV(ref_point=ref_point)
print('\nMin-Max normalization -> Min: ', obj_min)
print('Min-Max normalization -> Max: ', obj_max)
print('HV Reference points: ', ref_point)

# Metrics: IGD+
n_points = 200
if problem_name == 'dtlz5':
    X_opt = np.full((n_points, n_var), 0.5)
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem.evaluate(X_opt)

elif problem_name == 'dtlz6':
    X_opt = np.zeros((n_points, n_var))
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem.evaluate(X_opt)

elif problem_name == 'dtlz7':
    X_opt = np.zeros((n_points, n_var))
    X_opt[:, :n_obj-1] = np.linspace(0, 1, n_points).reshape(-1, 1)
    pf = problem.evaluate(X_opt)
else:
    pf = problem.pareto_front()
igd_plus = IGDPlus(pf)

Problem name: dtlz5
Cons: 0
Var: 10
Obj: 2

Sampling X_train shape:  (109, 10)
y_val shape:  (100, 2)
y_test shape:  (100, 2)

Min-Max normalization -> Min:  [0 0]
Min-Max normalization -> Max:  [2.61 2.7 ]
HV Reference points:  [1.1 1.1]


###### 2. BNN model training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
scaler_y_f1 = StandardScaler()
scaler_y_f2 = StandardScaler()
y_train_f1_scaled = scaler_y_f1.fit_transform(y_train_f1.reshape(-1, 1)).flatten()
y_train_f2_scaled = scaler_y_f2.fit_transform(y_train_f2.reshape(-1, 1)).flatten()
y_val_f1_scaled = scaler_y_f1.transform(y_val[:, 0].reshape(-1, 1)).flatten()
y_val_f2_scaled = scaler_y_f2.transform(y_val[:, 1].reshape(-1, 1)).flatten()

model_f1, guide_f1 = train_bnn(
    torch.tensor(X_train, dtype=torch.float32).to(device),
    torch.tensor(y_train_f1_scaled, dtype=torch.float32).to(device),
    torch.tensor(X_val, dtype=torch.float32).to(device),
    torch.tensor(y_val_f1_scaled, dtype=torch.float32).to(device),
    hidden=16,
    max_steps=50000,
    lr=1e-3,
    patience=10)

model_f2, guide_f2 = train_bnn(
    torch.tensor(X_train, dtype=torch.float32).to(device),
    torch.tensor(y_train_f2_scaled, dtype=torch.float32).to(device),
    torch.tensor(X_val, dtype=torch.float32).to(device),
    torch.tensor(y_val_f2_scaled, dtype=torch.float32).to(device),
    hidden=16,
    max_steps=50000,
    lr=1e-3,
    patience=10)

cpu
[200] train ELBO=2.95e+07, val MSE=5.48e-01, no_improve=0/10
[400] train ELBO=1.45e+07, val MSE=2.58e-01, no_improve=0/10
[600] train ELBO=1.16e+07, val MSE=1.84e-01, no_improve=0/10
[800] train ELBO=1.67e+07, val MSE=1.66e-01, no_improve=0/10
[1000] train ELBO=8.99e+06, val MSE=1.63e-01, no_improve=0/10
[1200] train ELBO=1.33e+07, val MSE=1.44e-01, no_improve=0/10
[1400] train ELBO=5.00e+07, val MSE=1.51e-01, no_improve=1/10
[1600] train ELBO=8.78e+06, val MSE=1.29e-01, no_improve=0/10
[1800] train ELBO=7.89e+06, val MSE=1.28e-01, no_improve=0/10
[2000] train ELBO=7.11e+06, val MSE=1.25e-01, no_improve=0/10
[2200] train ELBO=1.88e+07, val MSE=1.41e-01, no_improve=1/10
[2400] train ELBO=6.34e+06, val MSE=1.23e-01, no_improve=0/10
[2600] train ELBO=8.47e+06, val MSE=1.22e-01, no_improve=0/10
[2800] train ELBO=5.53e+06, val MSE=1.14e-01, no_improve=0/10
[3000] train ELBO=8.84e+06, val MSE=1.21e-01, no_improve=1/10
[3200] train ELBO=5.37e+06, val MSE=1.05e-01, no_improve=0/10
[3400] t

###### 2.1 MSE CICP

In [ ]:
y1_q50, y1_q80, y1_q90, y1_q95 = predict_bnn(model_f1, guide_f1, X_test, num_samples=100)
y2_q50, y2_q80, y2_q90, y2_q95 = predict_bnn(model_f2, guide_f2, X_test, num_samples=100)
y1_q50 = scaler_y_f1.inverse_transform(y1_q50)
y1_q80 = scaler_y_f1.inverse_transform(y1_q80)
y1_q90 = scaler_y_f1.inverse_transform(y1_q90)
y1_q95 = scaler_y_f1.inverse_transform(y1_q95)
y2_q50 = scaler_y_f2.inverse_transform(y2_q50)
y2_q80 = scaler_y_f2.inverse_transform(y2_q80)
y2_q90 = scaler_y_f2.inverse_transform(y2_q90)
y2_q95 = scaler_y_f2.inverse_transform(y2_q95)

y_q50 = np.concatenate([y1_q50, y2_q50], axis=1)
y_q80 = np.concatenate([y1_q80, y2_q80], axis=1)
y_q90 = np.concatenate([y1_q90, y2_q90], axis=1)
y_q95 = np.concatenate([y1_q95, y2_q95], axis=1)

# Metrics
mse_bnn = mean_squared_error(y_test, y_q50)
print(f"BNN MSE: {mse_bnn:.2e}\n")

print('y_q50\n', y_q50[0:3])
print('y_q80\n', y_q80[0:3])
print('y_q90\n', y_q90[0:3])
print('y_q95\n', y_q95[0:3])

BNN MSE: 3.04e-02

y_q50
 [[0.682  1.4412]
 [1.5092 0.855 ]
 [1.6621 0.393 ]]
y_q80
 [[0.8069 1.497 ]
 [1.5706 0.9471]
 [1.7268 0.4786]]
y_q90
 [[0.8562 1.5204]
 [1.6052 0.9757]
 [1.7407 0.5301]]
y_q95
 [[0.894  1.5401]
 [1.6385 1.0271]
 [1.7844 0.5557]]


In [ ]:
###### CICP
"""
Interval    Coverage
±1.282σ      80.00%
±1.645σ      90.00%
±1.960σ      95.00%
"""
def coverage(y_test, y_q50, y_upper):
    err = np.abs(y_test - y_q50)
    inside = y_test <= y_upper
    per_dim = inside.mean(axis=0)
    overall = inside.mean()
    return per_dim, overall
print('Coverage')
for level, q_upper in [(0.90, y_q90)]:
    per_dim, overall = coverage(y_test, y_q50, q_upper)
    print(f"{int(level*100)}% interval: per_dim={per_dim*100}%, overall={overall*100:.1f}%")

Coverage
90% interval: per_dim=[79. 79.]%, overall=79.0%


###### 3. Optimization using BNN

In [ ]:
# Algorithm
algorithm = NSGA2(
    pop_size=100,
    crossover = SBX(prob=1.0, eta=20),
    mutation = PM(prob=1/n_var, eta=20),
    survival=Survival_standard(),     ### change
    eliminate_duplicates=True)

mse_list, igd_list, hv_surrogate_list, hv_real_list,  = [], [], [], []

for seed in range(1, 2):
    benchmark_problem = Benchmark_Problem(
        model_f1=model_f1,
        model_f2=model_f2,
        guide_f1=guide_f1,
        guide_f2=guide_f2,
        scaler_y_f1=scaler_y_f1,
        scaler_y_f2=scaler_y_f2,
        n_var=n_var,
        n_obj=n_obj,
        xl=problem.xl,
        xu=problem.xu,
        problem_name=problem_name,
        use_surrogate='BNN_uncertainty')


    # Optimization
    start_time = time.time()

    res = minimize(
        benchmark_problem,
        algorithm,
        termination = get_termination("n_gen", n_gen),
        seed=seed,
        save_history=True,
        verbose=False)

    end_time = time.time()

    solution = res.history[-1].opt.get("X")
    obj = res.history[-1].opt.get("F")
    f_real = problem.evaluate(solution, return_values_of=["F"])

    # MSE
    mse = mean_squared_error(f_real, obj)
    mse_list.append(mse)
    # IGD+
    igd_plus_real = float(igd_plus(f_real))
    igd_list.append(igd_plus_real)
    # HV
    f_real_normalization = (f_real - obj_min) / (obj_max - obj_min)
    obj_normalization = (obj - obj_min) / (obj_max - obj_min)
    hv_real = float(hv.do(f_real_normalization))
    hv_surrogate = float(hv.do(obj_normalization))
    hv_real_list.append(hv_real)
    hv_surrogate_list.append(hv_surrogate)

    max_obj = np.max(obj, axis=0)
    max_obj_real = np.max(f_real, axis=0)
    print(f"Seed {seed} | Time: {end_time - start_time:.2f}s | "
          f"MSE: {mse:.2e} | "
          f"igd+: {igd_plus_real:.2e} | "
          f"Sur HV: {hv_surrogate:.2f} | "
          f"Real HV: {hv_real:.2f} | "
          f"Max f_real: {max_obj_real}")

plot_obj_2d(obj,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))
plot_obj_2d(f_real,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))

Seed 1 | Time: 45.02s | MSE: 3.63e-01 | igd+: 8.57e-01 | Sur HV: 0.93 | Real HV: 0.77 | Max f_real: [2.5074 2.473 ]


In [ ]:
# mean_mse, std_mse = mean_std(mse_list)
# mean_igd, std_igd = mean_std(igd_list)
# mean_hv_real, std_hv_real = mean_std(hv_real_list)
# mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

# print('Problem name: ', problem_name)
# print("\n=== BNN ===")

# print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
# print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
# print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
# print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")

###### 3. Optimization using BNN + dual-ranking (q=0.90)

In [ ]:
# Algorithm
algorithm = NSGA2(
    pop_size=100,
    crossover = SBX(prob=1.0, eta=20),
    mutation = PM(prob=1/n_var, eta=20),
    survival=Survival_fix_dual_ranking(alpha=0.9),     ### change
    eliminate_duplicates=True)

mse_list, igd_list, hv_surrogate_list, hv_real_list,  = [], [], [], []

for seed in range(1, 2):
    benchmark_problem = Benchmark_Problem(
        model_f1=model_f1,
        model_f2=model_f2,
        guide_f1=guide_f1,
        guide_f2=guide_f2,
        scaler_y_f1=scaler_y_f1,
        scaler_y_f2=scaler_y_f2,
        n_var=n_var,
        n_obj=n_obj,
        xl=problem.xl,
        xu=problem.xu,
        problem_name=problem_name,
        use_surrogate='BNN_uncertainty')

    # Optimization
    start_time = time.time()

    res = minimize(
        benchmark_problem,
        algorithm,
        termination = get_termination("n_gen", n_gen),
        seed=seed,
        save_history=True,
        verbose=False)

    end_time = time.time()

    solution_q90 = res.history[-1].opt.get("X")
    obj_q90 = res.history[-1].opt.get("F")
    f_real_q90 = problem.evaluate(solution_q90, return_values_of=["F"])

    # MSE
    mse = mean_squared_error(f_real_q90, obj_q90)
    mse_list.append(mse)
    # IGD+
    igd_plus_real = float(igd_plus(f_real_q90))
    igd_list.append(igd_plus_real)
    # HV
    f_real_normalization = (f_real_q90 - obj_min) / (obj_max - obj_min)
    obj_normalization = (obj_q90 - obj_min) / (obj_max - obj_min)
    hv_real = float(hv.do(f_real_normalization))
    hv_surrogate = float(hv.do(obj_normalization))
    hv_real_list.append(hv_real)
    hv_surrogate_list.append(hv_surrogate)

    max_obj = np.max(obj_q90, axis=0)
    max_obj_real = np.max(f_real_q90, axis=0)
    print(f"Seed {seed} | Time: {end_time - start_time:.2f}s | "
          f"MSE: {mse:.2e} | "
          f"igd+: {igd_plus_real:.2e} | "
          f"Sur HV: {hv_surrogate:.2f} | "
          f"Real HV: {hv_real:.2f} | "
          f"Max f_real: {max_obj_real}")

plot_obj_2d(obj_q90,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))
plot_obj_2d(f_real_q90,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))

Seed 1 | Time: 44.86s | MSE: 5.13e-01 | igd+: 1.11e+00 | Sur HV: 0.93 | Real HV: 0.66 | Max f_real: [2.6985 2.6262]


In [ ]:
# mean_mse, std_mse = mean_std(mse_list)
# mean_igd, std_igd = mean_std(igd_list)
# mean_hv_real, std_hv_real = mean_std(hv_real_list)
# mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

# print('Problem name: ', problem_name)
# print("\n=== BNN + f_fix + dual-ranking + q90 ===")

# print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
# print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
# print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
# print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")

###### Test

In [ ]:
result1 = evaluate_pre_real(y_q50, y_test, title="Test dataset")
result2 = evaluate_pre_real(obj, f_real, title="Opt f_sur vs. f_real")
result3 = evaluate_pre_real(obj_q90, f_real_q90, title="Opt dual-ranking(q=0.9) f_sur vs. f_real")

In [ ]:
f_real_max = np.max(f_real, axis=0)
print('f_real_max', f_real_max)

obj_max = np.max(obj, axis=0)
print('obj_max', obj_max)

def HV_calculation(f, problem_name):
    if problem_name == 'dtlz1':
      obj_min = np.array([0,0])
      obj_max = np.array([630,740])
    f_normalization = (f - obj_min) / (obj_max - obj_min)

    ref_point = np.array([1.1,1.1])
    hv = HV(ref_point=ref_point)
    result = float(hv.do(f_normalization))
    return result

print('obj', HV_calculation(obj, problem_name))
print('f_real', HV_calculation(f_real, problem_name))

print('obj_q90', HV_calculation(obj_q90, problem_name))
print('f_real_q90', HV_calculation(f_real_q90, problem_name))
